In [1]:
import json
import hashlib
import re
import sys
from urllib.request import build_opener, ProxyHandler

MODELO = "ministral-3:8b-instruct-2512-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Leer los dos archivos compartidos, un nivel arriba.
with open("../configuracion_v3.json", "rb") as archivo:
    contenido_configuracion = archivo.read()

with open("../casos_comparacion_v1.json", "rb") as archivo:
    contenido_casos = archivo.read()

configuracion = json.loads(contenido_configuracion)
casos = json.loads(contenido_casos)

# Comprobar que los casos mantienen su contenido original.
sha_casos = hashlib.sha256(contenido_casos).hexdigest()
sha_configuracion = hashlib.sha256(contenido_configuracion).hexdigest()

if sha_casos != configuracion["cases_sha256"]:
    raise ValueError("La huella del archivo de casos no coincide. No continuar.")

ids_esperados = [f"S-{numero:03d}" for numero in range(5, 11)]
if [caso["case_id"] for caso in casos] != ids_esperados:
    raise ValueError("Los identificadores o el orden de los casos cambiaron.")

if configuracion["api_url"] != API_LOCAL + "/api/chat":
    raise ValueError("La configuración no apunta a la API local esperada.")

# Conexión directa al servidor local, sin utilizar proxies.
conexion_local = build_opener(ProxyHandler({}))

with conexion_local.open(API_LOCAL + "/api/version", timeout=10) as conexion:
    version_servidor = json.load(conexion)["version"]

# La ficha de esta variante indica soporte desde Ollama 0.13.1.
version_numerica = re.match(r"^v?(\d+)\.(\d+)\.(\d+)", version_servidor)
if version_numerica is None:
    raise RuntimeError("No se pudo interpretar la versión del servidor.")

if tuple(map(int, version_numerica.groups())) < (0, 13, 1):
    raise RuntimeError("El servidor es anterior a Ollama 0.13.1. No continuar.")

with conexion_local.open(API_LOCAL + "/api/tags", timeout=10) as conexion:
    inventario = json.load(conexion)["models"]

modelo_disponible = any(
    modelo.get("name") == MODELO for modelo in inventario
)

print("Python del kernel:", sys.version.split()[0])
print("Ollama — servidor:", version_servidor)
print("Casos verificados:", len(casos))
print("Identificadores:", [caso["case_id"] for caso in casos])
print("SHA256 de casos:", sha_casos)
print("SHA256 de configuración:", sha_configuracion)
print("Prompt:", configuracion["prompt_version"])
print("Parámetros:", configuracion["request_defaults"]["options"])
print("Modelo exacto ya disponible:", modelo_disponible)

Python del kernel: 3.11.16
Ollama — servidor: 0.40.13
Casos verificados: 6
Identificadores: ['S-005', 'S-006', 'S-007', 'S-008', 'S-009', 'S-010']
SHA256 de casos: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
SHA256 de configuración: 1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af
Prompt: V3
Parámetros: {'num_ctx': 4096, 'num_predict': 600, 'temperature': 0}
Modelo exacto ya disponible: False


In [2]:
import copy
import json
import time
from urllib.request import Request

# Caso diagnóstico, separado de los seis casos de comparación
contenido_diagnostico = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Reutilizar la configuración V3 sin modificar el archivo original
solicitud_ministral = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud_ministral["model"] = MODELO
solicitud_ministral["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id="DIAG-M01",
            contenido=contenido_diagnostico
        )
    }
]

# Preparar la solicitud a Ollama local
peticion = Request(
    configuracion["api_url"],
    data=json.dumps(
        solicitud_ministral,
        ensure_ascii=False
    ).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

# Ejecutar y conservar la respuesta original en memoria
inicio = time.perf_counter()

with conexion_local.open(
    peticion,
    timeout=configuracion["timeout_s"]
) as conexion:
    respuesta_ministral = json.load(conexion)

tiempo_cliente = time.perf_counter() - inicio

# Mostrar resultado
print("MODELO:", respuesta_ministral["model"])
print("\nRESPUESTA:\n")
print(respuesta_ministral["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_cliente, 3), "s")

for campo in (
    "total_duration",
    "load_duration",
    "prompt_eval_duration",
    "eval_duration"
):
    valor = respuesta_ministral.get(campo)
    if valor is not None:
        print(campo, round(valor / 1e9, 3), "s")

for campo in (
    "prompt_eval_count",
    "prompt_eval_cached_count",
    "eval_count",
    "done_reason"
):
    print(campo, respuesta_ministral.get(campo))


MODELO: ministral-3:8b-instruct-2512-q4_K_M

RESPUESTA:

**Evidencia:**
- **[E1]** Autor X ha publicado un artículo en el ámbito de las *bases de datos académicas* (ej.: gestión, diseño o análisis de repositorios).
- **[E2]** Autor Y ha publicado un artículo en el ámbito de la *recuperación de información* (ej.: algoritmos, métricas o sistemas de búsqueda).

---

**Interpretación:**
La predicción de coautoría favorable se sustenta en la **complementariedad temática documentada** entre los campos de investigación de ambos autores. Las bases de datos académicas y la recuperación de información son áreas interdependientes: los sistemas de recuperación requieren infraestructuras de datos (bases de datos) para operar, y los repositorios académicos dependen de técnicas de recuperación para ser accesibles. La evidencia **[E1] y [E2]** refleja una **cobertura de temas afines** que, en contextos académicos, suele asociarse a sinergias colaborativas, aunque no implique una interacción previa doc

In [3]:
import copy
import json
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

# 1. Verificar nuevamente los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

sha_casos_actual = hashlib.sha256(bytes_casos).hexdigest()
sha_config_actual = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos_actual == configuracion["cases_sha256"]
assert sha_config_actual == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

casos_prueba = json.loads(bytes_casos)
config_prueba = json.loads(bytes_config)

assert len(casos_prueba) == 6
assert [c["case_id"] for c in casos_prueba] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

# 2. Obtener el identificador verificable del modelo local
with conexion_local.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos_instalados = json.load(r)["models"]

modelo_info = next(
    (m for m in modelos_instalados if m["name"] == MODELO),
    None
)

if modelo_info is None:
    raise RuntimeError("No se encontró la variante exacta de Ministral.")

with conexion_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

# 3. Crear un solo archivo, sin sobrescribir resultados previos
nombre_resultados = "resultados.jsonl"
experimento_id = str(uuid4())

with open(nombre_resultados, "x", encoding="utf-8") as salida:

    def guardar_registro(registro):
        salida.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        salida.flush()

    # Conservar el diagnóstico previo si sigue en memoria
    if all(nombre in globals() for nombre in (
        "solicitud_ministral",
        "respuesta_ministral",
        "tiempo_cliente"
    )):
        diagnostico = {
            "execution_id": str(uuid4()),
            "experiment_id": experimento_id,
            "case_id": "DIAG-M01",
            "grupo": "diagnostico",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": version_ollama,
            "solicitud": solicitud_ministral,
            "respuesta_original": respuesta_ministral,
            "tiempo_cliente_s": round(tiempo_cliente, 3),
            "estado": "diagnostico",
            "fecha_registro_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        guardar_registro(diagnostico)
        print("DIAG-M01 | diagnóstico conservado")

    # 4. Ejecutar los seis casos sin modificar V3
    for caso in casos_prueba:

        solicitud = copy.deepcopy(
            config_prueba["request_defaults"]
        )

        solicitud["model"] = MODELO
        solicitud["messages"] = [
            {
                "role": "system",
                "content": config_prueba["system_prompt"]
            },
            {
                "role": "user",
                "content": config_prueba[
                    "user_prompt_template"
                ].format(
                    case_id=caso["case_id"],
                    contenido=caso["contenido"]
                )
            }
        ]

        registro = {
            "execution_id": str(uuid4()),
            "experiment_id": experimento_id,
            "case_id": caso["case_id"],
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "prompt_version": config_prueba["prompt_version"],
            "modelo_digest": modelo_info["digest"],
            "modelo_detalles": modelo_info.get("details"),
            "ollama_version": version_ollama,
            "sha256_casos": sha_casos_actual,
            "sha256_configuracion": sha_config_actual,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config_prueba["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with conexion_local.open(
                peticion,
                timeout=config_prueba["timeout_s"]
            ) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif (
                respuesta.get("done") is True
                and respuesta.get("done_reason") == "stop"
                and respuesta.get("message", {}).get("content", "")
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )
        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar inmediatamente, incluso si hubo error
        guardar_registro(registro)

        print(
            caso["case_id"],
            "|", registro["estado"],
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo creado:", nombre_resultados)
print("Casos de comparación procesados:", len(casos_prueba))


DIAG-M01 | diagnóstico conservado
S-005 | truncada | 9.431 s
S-006 | ok | 8.683 s
S-007 | truncada | 8.389 s
S-008 | ok | 8.126 s
S-009 | truncada | 8.503 s
S-010 | truncada | 8.879 s

Archivo creado: resultados.jsonl
Casos de comparación procesados: 6


In [4]:
import json
import copy
import hashlib
import time
from datetime import datetime, timezone
from uuid import uuid4
from urllib.request import Request, build_opener, ProxyHandler

# 1. Verificar los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    datos_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    datos_config = f.read()

casos = json.loads(datos_casos)
config = json.loads(datos_config)

sha_casos = hashlib.sha256(datos_casos).hexdigest()
sha_config = hashlib.sha256(datos_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

# 2. Recuperar las ejecuciones originales
with open("resultados.jsonl", encoding="utf-8") as f:
    anteriores = [json.loads(line) for line in f if line.strip()]

originales = {
    r["case_id"]: r for r in anteriores
    if r.get("grupo") == "comparacion"
    and r.get("ronda") is None
}

assert set(originales) == {c["case_id"] for c in casos}

# Verificar que no cambió la variante instalada
modelo = "ministral-3:8b-instruct-2512-q4_K_M"
cliente = build_opener(ProxyHandler({}))
api_local = "http://127.0.0.1:11434"

with cliente.open(api_local + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

actual = next(m for m in instalados if m["name"] == modelo)

for r in originales.values():
    assert r["modelo_digest"] == actual["digest"]
    assert r["sha256_casos"] == sha_casos
    assert r["sha256_configuracion"] == sha_config

# Evitar repetir casos ya registrados en esta ronda
completados = {
    r["case_id"] for r in anteriores
    if r.get("ronda") == "R2_1200"
}

# 3. Ejecutar y añadir registros al mismo JSONL
with open("resultados.jsonl", "a", encoding="utf-8") as salida:

    for caso in casos:
        identificador = caso["case_id"]

        if identificador in completados:
            print(identificador, "| ya registrado")
            continue

        solicitud = copy.deepcopy(
            originales[identificador]["solicitud"]
        )

        # Verificar que el prompt original no cambió
        assert solicitud["messages"][0]["content"] == config["system_prompt"]
        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=identificador,
                contenido=caso["contenido"]
            )
        )
        assert solicitud["options"]["num_predict"] == 600

        # Único cambio experimental
        solicitud["options"]["num_predict"] = 1200

        registro = {
            "execution_id": str(uuid4()),
            "case_id": identificador,
            "grupo": "comparacion",
            "ronda": "R2_1200",
            "motivo_ajuste": "Truncamientos con limite de 600 tokens",
            "prompt_version": "V3",
            "modelo_digest": actual["digest"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(
                peticion, timeout=config["timeout_s"]
            ) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif (
                respuesta.get("done_reason") == "stop"
                and respuesta.get("message", {}).get("content")
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )
        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        salida.write(json.dumps(registro, ensure_ascii=False) + "\n")
        salida.flush()

        respuesta = registro.get("respuesta_original", {})

        print(
            identificador,
            "|", registro["estado"],
            "|", respuesta.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado: resultados.jsonl")


S-005 | ok | 796 tokens | 16.163 s
S-006 | ok | 461 tokens | 8.43 s
S-007 | ok | 755 tokens | 13.871 s
S-008 | ok | 650 tokens | 10.449 s
S-009 | ok | 700 tokens | 10.775 s
S-010 | ok | 586 tokens | 9.054 s

Archivo actualizado: resultados.jsonl
